# 🧠 IntelliCode-SL | Classifier SLM Fine-Tuning
**Model:** `Qwen2.5-Coder-0.5B-Instruct`  
**Task:** Classify user prompt → `debug | generate | modify | explain | document`

> Run cells top to bottom. GPU runtime required (Runtime → Change runtime type → T4 GPU)

In [ ]:
# ── Cell 1: Install Dependencies ──────────────────────────────
!pip install -q unsloth transformers datasets peft accelerate bitsandbytes trl
print("✅ Dependencies installed")

In [ ]:
# ── Cell 2: Imports ───────────────────────────────────────────
import os, torch
from google.colab import drive
from huggingface_hub import login
from datasets import load_dataset
from transformers import TrainingArguments
from unsloth import FastLanguageModel
from trl import SFTTrainer
print("✅ Imports done | GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT FOUND")

In [ ]:
# ── Cell 3: Mount Google Drive ────────────────────────────────
drive.mount("/content/drive")

DRIVE_BASE   = "/content/drive/MyDrive/IntelliCode-SL"
DATASET_PATH = f"{DRIVE_BASE}/datasets/classifier_dataset.json"
ADAPTER_SAVE = f"{DRIVE_BASE}/adapters/classifier_adapter"

os.makedirs(ADAPTER_SAVE, exist_ok=True)
print(f"✅ Drive mounted")
print(f"   Dataset path : {DATASET_PATH}")
print(f"   Adapter save : {ADAPTER_SAVE}")

In [ ]:
# ── Cell 4: HuggingFace Login ──────────────────────────────────
# Get your token from: https://huggingface.co/settings/tokens
HF_TOKEN = "hf_XXXXXXXXXXXXXXXXXXXXXXXXXX"   # ← paste your token here

login(token=HF_TOKEN)
print("✅ Logged in to HuggingFace")

In [ ]:
# ── Cell 5: Load Model via Unsloth ─────────────────────────────
MAX_SEQ_LEN = 512

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "Qwen/Qwen2.5-Coder-0.5B-Instruct",
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
    token          = HF_TOKEN,
)
print("✅ Model loaded")

In [ ]:
# ── Cell 6: Attach LoRA Adapter ────────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    lora_alpha     = 16,
    lora_dropout   = 0.05,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 42,
)
print("✅ LoRA adapter attached")
model.print_trainable_parameters()

## 📁 Dataset Format
Your `classifier_dataset.json` in Drive should look like:
```json
[
  {"instruction": "Fix the IndexError in my code", "label": "debug"},
  {"instruction": "Write a function to reverse a string", "label": "generate"},
  {"instruction": "Add type hints to this function", "label": "modify"},
  {"instruction": "What does this code do?", "label": "explain"},
  {"instruction": "Generate docstrings for this class", "label": "document"}
]
```
**Valid labels:** `debug` | `generate` | `modify` | `explain` | `document`

In [ ]:
# ── Cell 7: Load & Format Dataset ──────────────────────────────
PROMPT_TEMPLATE = """### Instruction:
Classify the following user request into exactly one category:
debug, generate, modify, explain, document

### User Request:
{}

### Category:
{}"""

EOS = tokenizer.eos_token

def format_sample(sample):
    return {"text": PROMPT_TEMPLATE.format(sample["instruction"], sample["label"]) + EOS}

raw_dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
dataset     = raw_dataset.map(format_sample)

print(f"✅ Dataset loaded: {len(dataset)} samples")
print("\nSample preview:")
print(dataset[0]["text"][:300])

In [ ]:
# ── Cell 8: Training Arguments ─────────────────────────────────
training_args = TrainingArguments(
    output_dir                  = "/content/classifier_checkpoints",
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 4,
    num_train_epochs            = 3,
    learning_rate               = 2e-4,
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    logging_steps               = 20,
    save_strategy               = "epoch",
    warmup_ratio                = 0.03,
    lr_scheduler_type           = "cosine",
    report_to                   = "none",
)
print("✅ Training args set")

In [ ]:
# ── Cell 9: Train ──────────────────────────────────────────────
trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset,
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LEN,
    args               = training_args,
)

print("🚀 Starting training...")
trainer.train()
print("✅ Training complete!")

In [ ]:
# ── Cell 10: Save Adapter to Google Drive ──────────────────────
model.save_pretrained(ADAPTER_SAVE)
tokenizer.save_pretrained(ADAPTER_SAVE)
print(f"✅ Adapter saved to Drive → {ADAPTER_SAVE}")

In [ ]:
# ── Cell 11: Quick Inference Test ──────────────────────────────
FastLanguageModel.for_inference(model)

test_cases = [
    "Fix the NullPointerException in my Java code",
    "Write a function to calculate fibonacci numbers",
    "Add error handling to this function",
    "What does this recursive function do?",
    "Generate docstrings for my Python class",
]

for prompt in test_cases:
    test_input = PROMPT_TEMPLATE.format(prompt, "")
    inputs = tokenizer(test_input, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=10, temperature=0.1)
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    label  = result.split("### Category:")[-1].strip().split()[0]
    print(f"  '{prompt[:45]}...' → {label}")